# Conditional alphas {#sec-alphas}

Now, we need to get the conditional alphas from the risk-adjusted returns and state variables.

The following math and commentary is taken largely as-is from the paper. First, I present the theory, then the implementation in Python, and finally, the results.

## Methodology

The methodology is taken from @Distaso2013, section 2.2.

In [src/betas.ipynb](src/betas.ipynb)/@sec-betas, we calculated the risk-adjusted returns $\hat{Z}_{i, t+1, M}$. This is useful because $Z_{i,t+1} = \alpha_{i,t} + \epsilon_{i,t+1} + \eta_{i,t+1}$. Identification of the conditional alpha stems from the fact that the conditional expectations of $\epsilon_{i,t+1}$ and $\eta_{i,t+1}$ are zero, so that $\alpha_{i,t} \equiv \mathbb{E}(Z_{i,t+1}|C_t)$.

Thus, it suffices to estimate the conditional expectation of the risk-adjusted returns, $\mathbb{E}(Z_{i,t+1}|C_t)$, to back out conditional alphas.

After estimating $\hat{Z}_{i, t+1, M}$ with the realized betas, we only need to estimate the conditional expectation $\mathbb{E}(\hat{Z}_{i,t+1,M}|C_t)$ of the realized risk-adjusted returns using nonparametric kernel methods. The paper proposes a consistent estimator with respect to the number of intraday observations $M$.

The high dimensionality of the state variable vector $C_t$ clashes with the nonparametric kernel estimator convergence, so we need to summarize it with PCA. Let then $\widehat{PC}_t = (\widehat{PC}_{1,t}, \dots, \widehat{PC}_{k,t})'$ denote the $k \le k_C$ principal component estimates of $C_t$.

The conditional alpha estimator is:

$$
\hat{\alpha}_{i,t} = \hat{m}_{i,T,M}(\widehat{PC}_t) = \frac{\frac{1}{T h_T^k} \sum_{\ell=1}^{T-1} \hat{Z}_{i,\ell+1,M} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_T}\right)}{\hat{g}_T(\widehat{PC}_t)}
$$

where $\hat{g}_T(\widehat{PC}_t) = \frac{1}{T h_T^k} \sum_{\ell=1}^{T-1} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_T}\right)$ is the kernel estimator of the density $g$ of the vector of principal components. 

To circumvent the difficulty of estimating $m_i(pc)$ over regions of low density, we consider a trimmed version of $\hat{m}_{i,T,M}(pc)$ as in Andrews (1995):

$$
tr_T\{\hat{m}_{i,T,M}(pc)\} = \hat{m}_{i,T,M}(pc) 1\{pc \in \hat{G}_T(pc)\}
$$

where $\hat{G}_T(pc) = \{c : \hat{g}_T(pc) > d_T\}$.

<!-- Instead of trimming, one could gauge the magnitude of the conditional alphas relative to the noise level by means of Treynor and Black’s (1973) appraisal ratio, which gauges the extra units of unsystematic risk investors have to bear for deviating from a passive management strategy. We explore this alternative in Appendix B.2. -->

### Expanding window calculation

The paper proposes an expanding window to restrict attention to real-time information: 

> To obtain conditional alphas, we entertain an expanding window to restrict attention to real-time information. We estimate the conditional expectation of the risk-adjusted returns using either an affine specification or a kernel-regression approach using a multivariate standard Gaussian kernel with rule-of-thumb bandwidths. As for the amount of trimming in (10), uniform consistency requires $d_T = O(h_T^\zeta)$ with $0 < \zeta < 1/4$. Note that, because $h_T \to 0$ as $T \to \infty$, trimming becomes more aggressive as $\zeta$ declines. We set $\zeta$ to 0.20 to enforce only a mild trimming, though we also experiment with other values of $\zeta$ in Section 4.2.

> The daily frequency of the analysis dictates the choice of state variables. In particular, we consider changes in the VIX index and in the Fed Fund rate, credit and term spreads, momentum, size and value factors, short- and long-term reversal factors, and volatility risk premium (namely, difference between the VIX index and realized volatility). To avoid the curse of dimensionality, we consider only their three first principal components.

This windowed version calculates $\widehat{PC}_t$ using data from $1:t$ only. And the bandwidth $h_T$ and trimming threshold $d_T$ are calculated separately for each $t$ using $t$ as $T$. Additionally, the kernel estimator does not uses data up to $T$, but only up to $t-1$. To differentiate the notation, let $W = t$ be the total window size:

$$
\hat{\alpha}_{i,t} = \hat{m}_{i,W,M}(\widehat{PC}_t) = \frac{\frac{1}{W h_W^k} \sum_{\ell=1}^{W-1} \hat{Z}_{i,\ell+1,M} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_W}\right)}{\hat{g}_W(\widehat{PC}_t)}
$$

<!-- CHECK: should it be fixed h_T and d_T still? -->

### Gaussian Kernel

For a single real-valued standardized deviation $u \in \mathbb{R}$, the standard univariate 1D Gaussian kernel function $K(u)$ is defined as the standard normal probability density function:

$$
K(u) = \frac{1}{\sqrt{2\pi}} \exp\left( -\frac{1}{2} u^2 \right)
$$

In our non-parametric setting, for a single feature $j$ at time $\ell$ evaluated at time $t$, $u_{\ell, j} = \frac{\widehat{PC}_{\ell, j} - \widehat{PC}_{t, j}}{h_W}$.

As established in Assumption _A(iv)_ of the paper, the $k$-dimensional joint kernel $K(u)$ across $k$ independent principal components $u = (u_1, u_2, \dots, u_k)'$ is constructed as a product kernel of $k$ univariate Gaussian kernels:

$$
K(u) = \prod_{j=1}^{k} K(u_j) = \prod_{j=1}^{k} \left[ \frac{1}{\sqrt{2\pi}} \exp\left( -\frac{1}{2} u_j^2 \right) \right]
$$

By factoring out the constant coefficient and applying the exponent product rule $\prod_{j=1}^{k} e^{a_j} = e^{\sum_{j=1}^{k} a_j}$, the joint kernel simplifies to:

$$
K(u) = \left( \frac{1}{\sqrt{2\pi}} \right)^k \prod_{j=1}^{k} \exp\left( -\frac{1}{2} u_j^2 \right) = \left( \frac{1}{\sqrt{2\pi}} \right)^k \exp\left( -\frac{1}{2} \sum_{j=1}^{k} u_j^2 \right)
$$


## The algorithm

... general comments about it, looping order, etc.



## Code setup

Core modules:

In [1]:
#| output: false
import os
if os.path.basename(os.getcwd()) == "src": os.chdir("..")

import numpy as np
import polars as pl

from pickle import dump as _dump
from scipy.io import loadmat as _loadmat
from tqdm import tqdm

from typing import NamedTuple, Final
from numpy.typing import NDArray

import src.utils as ut
from src.parameters import PARAMETERS as PARS

class sio:
    loadmat = staticmethod(_loadmat)

class pkl:
    dump = staticmethod(_dump)

Parameters:

In [2]:
class AlphasParameters(NamedTuple):
    window_warmup: int # Initial burn-in window size
    scaler: float      # Bandwidth scaling factor
    trim_exp: float    # Trimming exponent zeta in (0, 1/4)
    k: int             # Number of principal components to retain


PARS_ALPHAS: Final[AlphasParameters] = AlphasParameters(
    window_warmup = 500,
    scaler = 1.0,
    trim_exp = 0.2,
    k = 3
)

DATE_START, DATE_END = (
    pl.lit(x.astype("datetime64[D]").astype(np.uint32))
    for x in [PARS.date_start, PARS.date_end]
)

## Data loading and validation

The paper states the following data sources:

> The daily frequency of the analysis dictates the choice of state variables. In particular, we consider changes in the VIX index and in the Fed Fund rate, credit and term spreads, momentum, size and value factors, short- and long-term reversal factors, and volatility risk premium (namely, difference between the VIX index and realized volatility). We gather the Fama-French, momentum and reversal factors from Kenneth French’s webpage at http://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html. Interest rates are from the FRED database, whereas the VIX index is from the Chicago Board Options Exchange. To avoid the curse of dimensionality, we consider only their three first principal components.

This data was saved into the matlab data file `data/states_raw/FactorData_2018.mat`.

The timestamp variables seem to be days from epoch `0000-01-01`. Matlab has 1-indexing for days, so we need to subtract `719_529` to align with the epoch `1970-01-01` being used in Python.

In [ ]:
def load_state_data(path: str) -> pl.DataFrame:
    state_mat = sio.loadmat(path)

    state_raws = []
    for key in ["factors", "ADSdata"]:
        mat = state_mat[key]
        data = (
            pl.DataFrame(mat[:, 1:])
            .with_columns(ts_day_ny = (mat[:, 0] - 719_529).astype(np.uint32))
            .select(pl.col("ts_day_ny"), pl.all().exclude("ts_day_ny"))
        )
        state_raws.append(data)

    data_state = (
        state_raws[0]
        .join(state_raws[1], on = "ts_day_ny", how = "inner")
        .filter(
            (pl.col("ts_day_ny") >= DATE_START)
            & (pl.col("ts_day_ny") <= DATE_END)
        )
        .with_columns(pl.all().fill_nan(None))
        .pipe(lambda df:
            df.select(
                col for col in df.columns
                if not (df[col].is_null().any() or df[col].is_nan().any())
            )
        )
        .tail(-1) # To match with adjusted returns' dropped 1st observation
    )

    return data_state

data_state = load_state_data("data/states_raw/FactorData_2018.mat")
data_state.write_parquet("data/states_raw/states_clean.parquet")

A first look at the data is as below. Note that it has different levels of decimal precision, something that comes from the different sources.

In [25]:
with pl.Config(tbl_cols = 8):
    display(data_state)

ts_day_ny,column_0,column_1,column_2,…,column_893,column_894,column_895,column_896
u32,f64,f64,f64,…,f64,f64,f64,f64
11325,-3.39,-0.32,-5.04,…,-0.790969,-0.790969,-0.790969,-0.790969
11326,0.37,-0.43,-0.62,…,-0.808035,-0.808035,-0.808035,-0.808035
11327,1.7,-0.09,1.9,…,-0.825037,-0.825037,-0.825037,-0.825037
11330,1.17,0.15,1.49,…,-0.875067,-0.875067,-0.875067,-0.875067
11331,-1.85,-0.03,0.44,…,-0.891353,-0.891353,-0.891353,-0.891353
…,…,…,…,…,…,…,…,…
17522,0.28,0.0,0.19,…,-0.045277,-0.04509,-0.04509,-0.045083
17526,0.35,0.0,0.69,…,-0.105409,-0.105166,-0.105166,-0.105156
17527,0.22,0.0,-0.1,…,-0.12069,-0.120432,-0.120432,-0.120422


Returns loading:

In [4]:
data_adjusted = pl.scan_parquet("data/stocks_1min_adjusted.parquet")

The state variables dataset must be validated:

- The date ranges in both datasets should be the same. Ideally both date vectors should be identical, but it is not a problem if not.
- There should be no duplicate values in the date vectors.
- There should be no nulls or NaNs in the data.
- All variables should be float64.

In [5]:
range_states = tuple(f(data_state["ts_day_ny"]) for f in [min, max])

range_returns = (
    data_adjusted
    .select(
        min = pl.col("ts_day_ny").min(),
        max = pl.col("ts_day_ny").max()
    )
    .collect()
    .row(0)
)

if range_states != range_returns:
    raise ValueError(
        f"State and returns data have different date ranges: "
        f"{range_states} vs {range_returns}"
    )

Some days are only present in the state variables or in the returns data. While it would be ideal to not have that, it is not a problem to drop them.

In [6]:
days_state = pl.from_epoch(data_state["ts_day_ny"], "d").to_numpy()

days_only_state = ~ np.isin(days_state, PARS.trading_days_final)
print(
    f"{np.sum(days_only_state)} days in state but not in returns:",
    ut.format_days_byyear(days_state[days_only_state])
)

days_only_returns = ~ np.isin(PARS.trading_days_final, days_state)
print(
    f"\n{np.sum(days_only_returns)} days in returns but not in state:",
    ut.format_days_byyear(PARS.trading_days_final[days_only_returns])
)

49 days in state but not in returns: 
- 2001: 06-08, 07-03, 11-23, 
- 2002: 07-05, 09-11, 11-29, 12-24, 
- 2003: 07-03, 08-15, 11-28, 12-24, 12-26, 
- 2004: 11-26, 
- 2005: 11-25, 12-23, 
- 2006: 07-03, 11-24, 
- 2007: 07-03, 11-23, 12-24, 
- 2008: 07-03, 11-28, 12-24, 12-26, 
- 2009: 11-27, 12-24, 
- 2010: 11-26, 12-27, 
- 2011: 08-15, 11-25, 12-23, 12-27, 
- 2012: 07-03, 11-23, 12-24, 
- 2013: 07-03, 08-22, 11-29, 12-24, 
- 2014: 07-03, 11-28, 12-24, 12-26, 
- 2015: 11-27, 12-24, 
- 2016: 11-25, 12-23, 
- 2017: 07-03, 11-24, 

40 days in returns but not in state: 
- 2001: 01-02, 03-08, 03-21, 10-08, 11-12, 
- 2002: 10-14, 10-31, 11-11, 
- 2003: 01-17, 01-21, 10-13, 11-11, 
- 2004: 01-12, 01-13, 10-11, 10-12, 11-11, 
- 2005: 10-10, 11-11, 
- 2006: 10-09, 
- 2007: 10-08, 11-12, 
- 2008: 10-13, 11-11, 
- 2009: 10-12, 11-11, 
- 2010: 10-11, 11-11, 
- 2011: 10-10, 11-11, 
- 2012: 10-08, 
- 2013: 10-14, 11-11, 
- 2014: 10-13, 11-11, 
- 2015: 11-11, 
- 2016: 10-10, 11-11, 
- 2017: 10-09, 11

Checking date duplicates:

In [7]:
if data_state["ts_day_ny"].is_duplicated().any():
    raise ValueError("Duplicate dates in state data")

Checking date types and nulls/NaNs:

In [8]:
if not all(
    x == pl.Float64
    for x in data_state.select(pl.all().exclude("ts_day_ny")).dtypes
):
    raise ValueError("State data contains non-float64 columns")

(
    any(data_state.select(f(pl.all().exclude("ts_day_ny")).any()).row(0))
    for f in [pl.Expr.is_nan, pl.Expr.is_null]
)

if any(
    any(data_state.select(f(pl.all().exclude("ts_day_ny")).any()).row(0))
    for f in [pl.Expr.is_nan, pl.Expr.is_null]
):
    raise ValueError("State data contains nulls or NaNs")

## Calculating window PCs

Let $W$ denote the total size of a window, which is equal to $t$.

In [ ]:
def pca_window(
    C_1_W: NDArray[np.float64], # (W, k) historical state variables
    k: int
) -> NDArray[np.float64]:
    # Standardization:
    mean = np.mean(C_1_W, axis = 0) # (k,) Across t
    std = np.std(C_1_W, axis = 0, ddof = 1) # (k,) Across t
    if np.any(std == 0.0):
        raise ValueError("Constant feature detected in PCA window.")

    C_1_W_norm = (C_1_W - mean) / std

    # Eigendecomposition:
    cov_mat = np.cov(C_1_W_norm, rowvar = False, ddof = 1) # (k, k)
    eigvals, eigvecs = np.linalg.eigh(cov_mat)

    # Principal components:
    idx_sort = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, idx_sort]
    scores = C_1_W_norm @ eigvecs[:, :k] # (T, k)

    # Rescaling:
    scores = scores / 100.0
    score_mean = np.mean(scores, axis = 0)
    score_std = np.std(scores, axis = 0, ddof = 1)
    if np.any(score_std == 0.0):
        raise ValueError("Constant feature detected in PCs window.")

    return (scores - score_mean) / score_std

# CHECK: if division by zero exist, should we just std[std == 0.0] = 1.0?

In [10]:
class WindowArgs(NamedTuple):
    t: int
    t_1_Wm1: NDArray[np.uint32] # (W,) indices of the window
    PC_1_Wm1: NDArray[np.float64]    # (W-1, k) historical PC predictors
    PC_t: NDArray[np.float64]        # (k,) current PC predictor at day t
    h_W: np.float64
    d_W: np.float64

def get_windows_data(
    data_state: pl.DataFrame
):
    windows_args = []
    ts = data_state["ts_day_ny"][(PARS_ALPHAS.window_warmup - 1):-1]
    # Including the PARS_ALPHAS.window_warmup's obs, and until T-1

    for t in tqdm(ts):
        data_state_w = data_state.filter(pl.col("ts_day_ny") <= t)

        t_1_Wm1 = data_state_w["ts_day_ny"].to_numpy()[:-1]
        window_data = data_state_w.select(pl.all().exclude("ts_day_ny")).to_numpy()

        # Principal components:
        W, k = window_data.shape[0], PARS_ALPHAS.k # CHECK: eq to scores.shape
        scores = pca_window(window_data, k)
        PC_1_Wm1 = scores[:-1, :] # Historical PC predictors
        PC_t = scores[-1, :] # Current PC predictor at day t

        # Kernel bandwidth:
        bandrate = (
            (4.0 / (k + 2.0)) ** (1.0 / (k + 4.0))
            * (W**(-1.0 / (k + 4.0)))
        )
        smx = np.sqrt(np.mean(np.std(scores, axis = 0, ddof = 1)**2))
        h_W = PARS_ALPHAS.scaler * bandrate * smx

        # Density trimming threshold:
        d_W = (bandrate * smx) ** PARS_ALPHAS.trim_exp

        windows_args.append((t, t_1_Wm1, PC_1_Wm1, PC_t, h_W, d_W))

    return windows_args

In [11]:
windows_args = get_windows_data(data_state)

with open("data/states_raw/windows_args.pkl", "wb") as f:
    pkl.dump(windows_args, f)

100%|██████████| 3733/3733 [00:31<00:00, 118.63it/s]


## Calculating conditional alphas

In [12]:
INV_SQRT_2PI_K: float = (
    (1.0 / np.sqrt(2.0 * np.pi))
    ** PARS_ALPHAS.k
)

def kernel_gauss(
    x: NDArray[np.float64]
) -> NDArray[np.float64]:
    return INV_SQRT_2PI_K * np.exp(-0.5 * np.sum(x**2, axis = 1)) # Across k

def cae_gauss_trim(
    PC_t: NDArray[np.float64],     # (k,) conditioning state variables at time t (PC_t)
    Z_2_W: NDArray[np.float64],    # (W-1,) target risk-adjusted returns (Z_{2:W})
    PC_1_Wm1: NDArray[np.float64], # (W-1, k) historical state variables (PC_{1:W-1})
    h_W: np.float64,               # Kernel bandwidth (h_W)
    d_W: np.float64                # Trimming density threshold (d_W)
) -> float:
    K_1_Wm1 = kernel_gauss((PC_1_Wm1 - PC_t) / h_W)  # (T-1,)
    deno = np.mean(K_1_Wm1) / (h_W ** PARS_ALPHAS.k) # Across t

    if deno <= d_W or np.isnan(deno):
        return 0.0

    num = np.mean(K_1_Wm1 * Z_2_W) # Across t
    return float(num / deno)
# We could use denominator cancellation to avoid some np.mean()

def cae_linear(
    PC_t: NDArray[np.float64],    # (k,)
    Z_2_W: NDArray[np.float64],   # (T-1,)
    PC_1_Wm1: NDArray[np.float64] # (T-1, k)
) -> np.float64:
    X = np.column_stack([np.ones(PC_1_Wm1.shape[0]), PC_1_Wm1])
    beta, _, _, _ = np.linalg.lstsq(X, Z_2_W)
    return beta[0] + PC_t @ beta[1:]

In [13]:
def get_permno_alphas(
    permno: int
):
    data_permno = (
        data_adjusted
        .filter(pl.col("permno") == permno)
        .select(["ts_day_ny", "adjusted_return"])
        .with_columns(Z_tp1 = pl.col("adjusted_return").shift(-1))
        .collect()
        .head(-1) # Remove null created by
    )

    t_permno = data_permno["ts_day_ny"].to_numpy()
    Z_2_Tp1 = data_permno["Z_tp1"].to_numpy()
    n = len(windows_args)

    res_ts = np.full(n, 0, dtype = np.uint32)
    res_alpha = np.full(n, np.nan, dtype = np.float64)
    res_linear = np.full(n, np.nan, dtype = np.float64)

    for i, w_args in enumerate(windows_args):
        t, t_1_Wm1, PC_1_Wm1, PC_t, h_W, d_W = w_args

        mask_t_pc_in_permno = np.isin(t_1_Wm1, t_permno, assume_unique = True)

        if not mask_t_pc_in_permno.sum() > PARS_ALPHAS.k:
            res_ts[i] = t
            continue

        idx_t_permno_in_pc = np.searchsorted(t_permno, t_1_Wm1[mask_t_pc_in_permno])
        Z_2_W = Z_2_Tp1[idx_t_permno_in_pc]
        PC_1_Wm1 = PC_1_Wm1[mask_t_pc_in_permno]

        # Nonparametric Kernel Alpha Estimate
        alpha_kernel = cae_gauss_trim(
            PC_t = PC_t, Z_2_W = Z_2_W, PC_1_Wm1 = PC_1_Wm1,
            h_W = h_W, d_W = d_W
        )

        # Linear Alpha Benchmark Estimate
        alpha_linear = cae_linear(
            PC_t = PC_t, Z_2_W = Z_2_W, PC_1_Wm1 = PC_1_Wm1
        )

        res_ts[i] = t
        res_alpha[i] = alpha_kernel
        res_linear[i] = alpha_linear

    return (permno, res_ts, res_alpha, res_linear)

In [14]:
#| execute: false
from src import alphas_mp

if 1: # Run only once
    permnos = (
        pl.read_csv("data/stocks_raw/counts_permno.csv")
        ['permno'].unique().to_list()
    )

    result_1min = alphas_mp.alphas_mp(
        permnos,
        (
            "data/states_raw/states_clean.parquet",
            "data/stocks_1min_adjusted.parquet",
            "data/states_raw/windows_args.pkl"
        ),
        max_workers = 6, chunksize = 10
    )
    result_1min.write_parquet("data/alphas_1min.parquet")

100%|██████████| 2364/2364 [10:43<00:00,  3.68it/s]


In [21]:
if 1: # Run only once
    result_5min = alphas_mp.alphas_mp(
        permnos,
        (
            "data/states_raw/states_clean.parquet",
            "data/stocks_5min_adjusted.parquet",
            "data/states_raw/windows_args.pkl"
        ),
        max_workers = 6, chunksize = 10
    )
    result_5min.write_parquet("data/alphas_5min.parquet")

100%|██████████| 2364/2364 [40:54<00:00,  1.04s/it]  


In [20]:
result_1min[["alpha", "linear_alpha"]].drop_nans().describe()

statistic,alpha,linear_alpha
str,f64,f64
"""count""",7.419924e6,7.419924e6
"""null_count""",0.0,0.0
"""mean""",-1.7561e-8,-0.000174
"""std""",0.000009,0.014517
"""min""",-0.003391,-17.494993
"""25%""",0.0,-0.000968
"""50%""",0.0,0.00003
"""75%""",0.0,0.000833
"""max""",0.002043,16.443149


In [22]:
result_5min[["alpha", "linear_alpha"]].drop_nans().describe()

statistic,alpha,linear_alpha
str,f64,f64
"""count""",7.419924e6,7.419924e6
"""null_count""",0.0,0.0
"""mean""",-2.1210e-8,-0.000316
"""std""",0.00001,0.01298
"""min""",-0.002674,-12.706597
"""25%""",0.0,-0.00117
"""50%""",0.0,-0.000115
"""75%""",0.0,0.000766
"""max""",0.002002,9.993612
